# OpenPlaque — Run optimized boundary refinement on new data

This notebook processes a new CCTA DICOM ZIP, such as the UCLA study, using the **current best-parameter JSON format** from boundary-parameter tuning.

It does **not** tune parameters. It loads:

```text
final_parameters_selected_on_all_cases
```

from `best_boundary_parameters_bayesian.json` or `best_boundary_parameters.json`, runs nnU-Net once per artery, applies the optimized boundary refinement, and writes TPV outputs.

Research use only. Not clinically validated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update OpenPlaque from GitHub
from pathlib import Path
import sys, os

REPO_DIR = Path('/content/OpenPlaque')
if not REPO_DIR.exists():
    !git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
else:
    !git -C /content/OpenPlaque pull

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Using OpenPlaque source:', SRC_DIR)

In [ ]:
# Install dependencies
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip install -q pandas scipy matplotlib SimpleITK pydicom

# Confirm GPU runtime
!nvidia-smi

In [ ]:
# Configure nnU-Net paths
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('nnUNet_results:', os.environ['nnUNet_results'])

In [ ]:
# Extract trained nnU-Net model if needed
from pathlib import Path
import zipfile

model_zip = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists():
    print('Model already extracted:', model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f'Missing model ZIP: {model_zip}')
    with zipfile.ZipFile(model_zip) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model to /content/nnUNet_results')

!find /content/nnUNet_results/Dataset001_CCTA_DHM -maxdepth 4 | head -80

In [ ]:
# Input paths for new-data inference
from pathlib import Path

STUDY_ZIP = Path('/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip')

# Keep the current JSON format from tuning; do not require extended metadata.
PARAMETER_JSON_CANDIDATES = [
    Path('/content/drive/MyDrive/OpenPlaque/Boundary_Parameter_Tuning/best_boundary_parameters_bayesian.json'),
    Path('/content/drive/MyDrive/OpenPlaque/Boundary_Parameter_Tuning/best_boundary_parameters.json'),
    Path('/content/drive/MyDrive/OpenPlaque/Boundary_Tuning/best_boundary_parameters_bayesian.json'),
    Path('/content/drive/MyDrive/OpenPlaque/Boundary_Tuning/best_boundary_parameters.json'),
]

BEST_PARAMETERS_JSON = next((p for p in PARAMETER_JSON_CANDIDATES if p.exists()), None)
if BEST_PARAMETERS_JSON is None:
    raise FileNotFoundError('Could not find best_boundary_parameters_bayesian.json or best_boundary_parameters.json in the expected Drive folders.')

OUTPUT_DIR = Path('/content/drive/MyDrive/OpenPlaque/New_Data_Optimized_Boundary')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Study ZIP:', STUDY_ZIP)
print('Best-parameter JSON:', BEST_PARAMETERS_JSON)
print('Output dir:', OUTPUT_DIR)

In [ ]:
# Load and display optimized parameters using the current JSON format
from openplaque.run_new_data import load_best_boundary_parameters

optimized_params = load_best_boundary_parameters(BEST_PARAMETERS_JSON)

print('Optimized boundary-refinement parameters')
print('-' * 60)
for k, v in optimized_params.items():
    print(f'{k}: {v}')

In [ ]:
# Load UCLA/new study and auto-detect artery series, falling back to known UCLA series
from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series

study = OpenPlaqueStudy(str(STUDY_ZIP))
study.summary()

series_map = auto_detect_or_fallback_series(study, fallback_series={'RCA': 1035, 'LCX': 1039, 'LAD': 1043})
print('Using artery series:', series_map)

In [ ]:
# Load LAD/RCA/LCX curved reformats
image_lad, volume_lad, _ = study.load_series(series_map['LAD'])
image_rca, volume_rca, _ = study.load_series(series_map['RCA'])
image_lcx, volume_lcx, _ = study.load_series(series_map['LCX'])

print('LAD:', volume_lad.shape, image_lad.GetSpacing())
print('RCA:', volume_rca.shape, image_rca.GetSpacing())
print('LCX:', volume_lcx.shape, image_lcx.GetSpacing())

In [ ]:
# Run nnU-Net segmentation once per artery
from openplaque.segmentation import segment_vessel

lad_report = segment_vessel(image_lad, volume_lad, 'LAD')
lad_report.summary()
lad_report.show_overlay(label=2)

rca_report = segment_vessel(image_rca, volume_rca, 'RCA')
rca_report.summary()
rca_report.show_overlay(label=2)

lcx_report = segment_vessel(image_lcx, volume_lcx, 'LCX')
lcx_report.summary()
lcx_report.show_overlay(label=2)

reports = [lad_report, rca_report, lcx_report]

In [ ]:
# Apply optimized boundary refinement; also compute conservative core masks
from openplaque.run_new_data import (
    refine_reports_with_parameters,
    core_reports_with_parameters,
    tpv_summary_rows,
)

refinements = refine_reports_with_parameters(reports, optimized_params)
core_results = core_reports_with_parameters(reports, optimized_params, erosion_iterations=1)
rows = tpv_summary_rows(reports, refinements, core_results=core_results)

for r in rows:
    print(r)

In [ ]:
# Visual QC of optimized refined masks
for report in reports:
    print('\n' + '=' * 80)
    print(report.name)
    refinements[report.name].summary()
    refinements[report.name].show_refined_overlay(report.volume)
    refinements[report.name].show_removed_overlay(report.volume)

In [ ]:
# Save CSV, refined masks, overlays, and HTML report
from openplaque.run_new_data import (
    save_tpv_summary_csv,
    save_refined_masks,
    save_overlays,
    write_new_data_html_report,
)

csv_path = save_tpv_summary_csv(rows, OUTPUT_DIR / 'new_data_tpv_summary.csv')
mask_paths = save_refined_masks(reports, refinements, OUTPUT_DIR / 'masks')
overlay_paths = save_overlays(reports, refinements, OUTPUT_DIR / 'overlays')
html_path = write_new_data_html_report(
    OUTPUT_DIR / 'new_data_optimized_tpv_report.html',
    rows,
    optimized_params,
    overlay_paths=overlay_paths,
)

print('Saved CSV:', csv_path)
print('Saved HTML:', html_path)
print('Saved masks:')
for p in mask_paths:
    print(' ', p)
print('Saved overlays:')
for p in overlay_paths:
    print(' ', p)